# Counting Worlds - サンプルデータを使ったデモ

**言語:** 日本語 | [English version](demo_en.ipynb)

このノートブックは、19世紀後半から20世紀初頭のナイジェリア新聞データを分析する「Counting Worlds」プロジェクトの中核機能を実演します。

## 概要

このデモでは以下の内容を扱います：

1. **統一データ読み込み関数**: 異なる種類の新聞データ（社説と読者投稿）を自動的にフォーマット判定して読み込める単一の関数
2. **データ構造の比較**: 社説データと読者投稿データのフォーマットの違いの理解
3. **基本的な統計分析**: 記事数、テキスト長、時間的分布などの指標の計算

## データソース

このプロジェクトでは、ナイジェリアの2つの主要新聞のデータを扱います：
- **Lagos Observer (LOE)**: 社説と読者投稿 (LOC)
- **Lagos Weekly Record (LWR/LWRE)**: 社説

このデモでは、`sample_data/` ディレクトリにある簡略化されたサンプルデータを使用します。

## 1. 必要なライブラリのインポート

**目的**: データ分析に必要なPythonライブラリをインポートします。

**各ライブラリの役割**:
- `pandas`: データ操作と分析のためのメインライブラリ
- `numpy`: 数値計算ライブラリ
- `os`: ファイルとパスの操作
- `re`: テキスト処理のための正規表現
- `datetime`: 日付と時刻の処理

In [1]:
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime

print("ライブラリが正常にインポートされました")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

ライブラリが正常にインポートされました
pandas version: 2.3.3
numpy version: 2.3.5


## 2. データ読み込み関数の定義

**目的**: 新聞データの読み込みと前処理のための中核関数を定義します。

### メイン関数: `load_newspaper_data()`

この関数は、異なる種類の新聞データを読み込むための統一的なインターフェースを提供します：

**主な機能**:
1. **自動タイプ検出**: ファイル名から社説か読者投稿かを自動判定
2. **自動ソース検出**: 新聞のソース（Lagos Observer、Lagos Weekly Recordなど）を自動識別
3. **カラム標準化**: 様々なカラム名を統一されたスキーマにマッピング
4. **メタデータ追加**: `data_source`と`article_type`カラムを追加
5. **特殊処理**: 読者投稿データに対する異なるロジック（「1_1」、「1_2」などの複合IDを処理）

**パラメータ**:
- `filepath`: CSVファイルへのパス
- `data_type`: オプション、'editorial'または'correspondence'を指定（Noneの場合は自動検出）
- `data_source`: オプション、ソース名を指定（Noneの場合は自動検出）

### ヘルパー関数: `preprocess_text()`

テキストデータを以下の処理でクリーニング：
- 小文字への変換
- アルファベット以外の文字を削除
- 空白文字の正規化

In [2]:
def load_newspaper_data(filepath, data_type=None, data_source=None):
    """
    統一的なナイジェリア新聞データ読み込み関数（改善版）
    
    Parameters:
    - filepath: CSVファイルパス
    - data_type: 'editorial'、'correspondence'、またはNone（自動判定）
    - data_source: データソース名、またはNone（自動判定）
    
    Returns:
    - 統一カラムとメタデータを持つDataFrame
    """
    import pandas as pd
    import os
    
    # CSVファイルを読み込み
    df = pd.read_csv(filepath, encoding='utf-8')
    
    filename = os.path.basename(filepath).lower()
    
    # ファイル名からデータタイプを自動判定（未指定の場合）
    if data_type is None:
        if 'loe' in filename:
            data_type = 'editorial'
        elif 'loc' in filename:
            data_type = 'correspondence'
        elif 'lwr' in filename or 'lwre' in filename:
            data_type = 'editorial'
        elif 'editorial' in filename:
            data_type = 'editorial'
        elif 'correspondence' in filename or 'letter' in filename:
            data_type = 'correspondence'
        else:
            data_type = 'editorial'  # デフォルト値
    
    # データソースを自動判定（未指定の場合）
    if data_source is None:
        if 'loe' in filename or 'loc' in filename:
            data_source = 'Lagos Observer'
        elif 'lwr' in filename or 'lwre' in filename:
            data_source = 'Lagos Weekly Record'
        elif 'lagos_observer' in filename or 'lo_' in filename:
            data_source = 'Lagos Observer'
        elif 'weekly_record' in filename or 'wr_' in filename:
            data_source = 'Lagos Weekly Record'
        else:
            # ファイルパスから意味のある名前を抽出
            basename = os.path.splitext(os.path.basename(filepath))[0]
            # 名前をクリーンアップ
            clean_name = basename.replace('_', ' ').title()
            data_source = f'Custom Source ({clean_name})'
    
    # メタデータカラムを追加
    df['data_source'] = data_source
    df['article_type'] = data_type
    
    # LOC特有のカラムマッピングを先に処理
    if data_type == 'correspondence':
        # LOC特有のマッピング調整
        if 'no' in df.columns and 'id_1' in df.columns:
            df['id'] = df['no']  # 通し番号をidに
            df['composite_id'] = df['id_1']  # 複合ID（1_1形式）を別列に保存
    
    # すべてのデータタイプに対する標準カラムマッピング
    column_mapping = {
        # テキストカラム
        'Text': 'text', 'TEXT': 'text',
        
        # 日付カラム - LOE/LWREのPublication Dateを含む
        'Date': 'date', 'DATE': 'date',
        'Publication Date': 'date', 'publication date': 'date',
        
        # 年カラム
        'Year': 'year', 'YEAR': 'year',
        
        # IDカラム（LOE/LWRE用、LOCには適用しない）
        'ID': 'id', 'Id': 'id', 'Article_ID': 'id', 'article_id': 'id'
    }
    
    for old_col, new_col in column_mapping.items():
        if old_col in df.columns:
            df.rename(columns={old_col: new_col}, inplace=True)
    
    # 必須カラムの存在を保証
    if 'id' not in df.columns:
        df['id'] = range(1, len(df) + 1)
    
    if 'year' not in df.columns and 'date' in df.columns:
        try:
            df['year'] = pd.to_datetime(df['date']).dt.year
        except:
            df['year'] = None
    
    return df


def preprocess_text(text):
    """
    テキスト前処理のヘルパー関数
    """
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    return text

## 3. 社説データの読み込み

**目的**: サンプル社説データを読み込み、内容を確認します。

**このセルが行うこと**:
1. `sample_data/sample_editorial.csv`から社説データを読み込み
2. 読み込んだデータの基本情報を表示：
   - レコード数
   - 検出されたデータソース
   - 記事タイプ
   - カラム数

**期待される出力**: 5件の社説レコードが正常に読み込まれたことを確認する情報。

In [3]:
print("=== 社説データの読み込み ===")
editorial_df = load_newspaper_data('./sample_data/sample_editorial.csv')

print(f"読み込み完了: {len(editorial_df)} レコード")
print(f"データソース: {editorial_df.data_source.iloc[0]}")
print(f"記事タイプ: {editorial_df.article_type.iloc[0]}")
print(f"カラム数: {len(editorial_df.columns)}")

=== 社説データの読み込み ===
読み込み完了: 5 レコード
データソース: Custom Source (Sample Editorial)
記事タイプ: editorial
カラム数: 7


## 4. 読者投稿データの読み込み

**目的**: サンプル読者投稿データを読み込み、内容を確認します。

**このセルが行うこと**:
1. `sample_data/sample_correspondence.csv`から読者投稿（読者の手紙）データを読み込み
2. 以下を含む基本情報を表示：
   - レコード数
   - 検出されたデータソース
   - 記事タイプ
   - カラム数
   - 複合IDの例（読者投稿特有のID形式）

**注意**: 読者投稿データには特殊なID形式（例：「1_1」、「1_2」）があります：
- 最初の数字: 号/日付のID
- 2番目の数字: その号内での手紙の番号

**期待される出力**: 5件の読者投稿レコードが正常に読み込まれたことを確認する情報。

In [4]:
print("=== 読者投稿データの読み込み ===")
correspondence_df = load_newspaper_data('./sample_data/sample_correspondence.csv')

print(f"読み込み完了: {len(correspondence_df)} レコード")
print(f"データソース: {correspondence_df.data_source.iloc[0]}")
print(f"記事タイプ: {correspondence_df.article_type.iloc[0]}")
print(f"カラム数: {len(correspondence_df.columns)}")
print(f"複合ID例: {correspondence_df.composite_id.tolist()[:3]}")

=== 読者投稿データの読み込み ===
読み込み完了: 5 レコード
データソース: Custom Source (Sample Correspondence)
記事タイプ: correspondence
カラム数: 9
複合ID例: ['1_1', '1_2', '1_3']


## 5. データ構造の比較

**目的**: 社説データと読者投稿データのカラム構造を調査・比較します。

**このセルが行うこと**:
1. 社説データの全カラムをリスト表示
2. 読者投稿データの全カラムをリスト表示
3. 各データセットからサンプル行を表示

**確認すべき主な違い**:

**社説データのカラム**:
- `id`: 一意の識別子
- `text`: 記事テキスト内容
- `date`: 発行日
- `year`: 日付から抽出した年
- `Years`: 年代（例：「1880s」）
- `data_source`: 新聞名
- `article_type`: 「editorial」

**読者投稿データのカラム**:
- `no`: 元の通し番号
- `id_1`: 複合ID（号_手紙番号形式）
- `text`: 手紙のテキスト内容
- `year`, `date`: 時間情報
- `data_source`, `article_type`: メタデータ
- `id`: `no`から変換
- `composite_id`: `id_1`から保存

**なぜ重要か**: これらの構造的な違いを理解することで、両方のデータタイプに対応する分析コードを書く際に役立ちます。

In [5]:
print("=== データ構造の比較 ===")
print("\n【社説データのカラム構造】")
for i, col in enumerate(editorial_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【読者投稿データのカラム構造】")
for i, col in enumerate(correspondence_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【社説データのサンプル】")
print(editorial_df.head())

print("\n【読者投稿データのサンプル】")
print(correspondence_df.head())

=== データ構造の比較 ===

【社説データのカラム構造】
 1. id
 2. text
 3. date
 4. year
 5. Years
 6. data_source
 7. article_type

【読者投稿データのカラム構造】
 1. no
 2. id_1
 3. text
 4. year
 5. date
 6. data_source
 7. article_type
 8. id
 9. composite_id

【社説データのサンプル】
   id                                               text        date  year  \
0   1  The government has announced new regulations f...  1882/03/02  1882   
1   2  Education remains a priority for the colonial ...  1882/03/09  1882   
2   3  The railway construction project continues to ...  1882/03/16  1882   
3   4  Local merchants express concerns about new tax...  1882/03/23  1882   
4   5  The Governor addressed the council on matters ...  1882/03/30  1882   

   Years                       data_source article_type  
0  1880s  Custom Source (Sample Editorial)    editorial  
1  1880s  Custom Source (Sample Editorial)    editorial  
2  1880s  Custom Source (Sample Editorial)    editorial  
3  1880s  Custom Source (Sample Editorial)    editorial  
4

## 6. 基本的な統計分析

**目的**: 読み込んだデータセットの基本統計を計算・表示します。

**このセルが行うこと**:

1. **データセット概要**:
   - 社説と読者投稿を別々にカウント
   - 総記事数を報告

2. **時間情報**:
   - 社説の年範囲を表示
   - 読者投稿の年範囲を表示
   - 利用可能な場合は年代情報を表示

3. **テキスト長分析**:
   - 各記事の文字数を計算
   - タイプ別の平均テキスト長を算出
   - 内容ボリュームの違いを理解するのに役立つ

4. **記事タイプ別の集計統計**:
   - すべてのデータを記事タイプでグループ化
   - テキスト長の件数、平均、標準偏差を表示
   - 時間範囲（最小/最大年）を表示

**使用例**: これらの統計は研究者が以下を理解するのに役立ちます：
- データセットの構成とバランス
- 時間的カバレッジ
- 典型的な記事/手紙の長さ
- 内容ボリュームのばらつき

In [6]:
print("=== 基本統計 ===")

print("\n【データセット概要】")
print(f"社説記事数: {len(editorial_df)}")
print(f"読者投稿数: {len(correspondence_df)}")
print(f"総記事数: {len(editorial_df) + len(correspondence_df)}")

print("\n【年代情報】")
print(f"社説の年: {editorial_df.year.unique()}")
print(f"社説の年代: {editorial_df.Years.unique() if 'Years' in editorial_df.columns else 'N/A'}")
print(f"読者投稿の年: {correspondence_df.year.unique()}")

print("\n【テキスト長の統計】")
editorial_df['text_length'] = editorial_df['text'].str.len()
correspondence_df['text_length'] = correspondence_df['text'].str.len()

print(f"社説の平均文字数: {editorial_df.text_length.mean():.1f} 文字")
print(f"読者投稿の平均文字数: {correspondence_df.text_length.mean():.1f} 文字")

print("\n【データタイプ別統計】")
combined_df = pd.concat([editorial_df, correspondence_df], ignore_index=True)
print(combined_df.groupby('article_type').agg({
    'text_length': ['count', 'mean', 'std'],
    'year': ['min', 'max']
}).round(1))

=== 基本統計 ===

【データセット概要】
社説記事数: 5
読者投稿数: 5
総記事数: 10

【年代情報】
社説の年: [1882]
社説の年代: ['1880s']
読者投稿の年: [1882]

【テキスト長の統計】
社説の平均文字数: 63.6 文字
読者投稿の平均文字数: 58.6 文字

【データタイプ別統計】
               text_length             year      
                     count  mean  std   min   max
article_type                                     
correspondence           5  58.6  6.7  1882  1882
editorial                5  63.6  4.2  1882  1882


## まとめ

このデモでは以下を示しました：

1. ✅ 統一された関数を使って異なる種類の新聞データを読み込む方法
2. ✅ 社説データと読者投稿データの構造的な違い
3. ✅ 新聞コーパスの基本的な統計分析技術

## 次のステップ

より高度な分析については、メインの分析ノートブックを参照してください：
- `Counting_Worlds_cleaned.ipynb`: 可視化とテキストマイニング技術を含む完全な分析